In [ ]:
pip install datarec-lib boto3 torch transformers faiss-cpu

  Using cached datarec_lib-1.5.7-py3-none-any.whl.metadata (1.3 kB)
  Using cached boto3-1.43.2-py3-none-any.whl.metadata (6.5 kB)
  Using cached torch-2.11.0-cp311-cp311-manylinux_2_28_x86_64.whl.metadata (29 kB)
  Using cached transformers-5.7.0-py3-none-any.whl.metadata (33 kB)
  Using cached faiss_cpu-1.13.2-cp310-abi3-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (7.6 kB)
  Using cached datarec-1.5.7-py3-none-any.whl.metadata (10 kB)
  Using cached botocore-1.43.2-py3-none-any.whl.metadata (5.5 kB)
  Using cached jmespath-1.1.0-py3-none-any.whl.metadata (7.6 kB)
  Using cached s3transfer-0.17.0-py3-none-any.whl.metadata (1.7 kB)
  Using cached filelock-3.29.0-py3-none-any.whl.metadata (2.0 kB)
  Using cached typing_extensions-4.15.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached cuda_toolkit-13.0.2-py2.py3-none-any.whl.metadata (9.4 kB)
  Using cached cuda_bindings-13.2.0-cp311-cp311-manylinux_2_24_x86_64.

In [ ]:
from datarec.datasets import Movielens
from io import BytesIO
import boto3
import pandas as pd
import numpy as np
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModel
import faiss

In [ ]:
s3 = boto3.client(
        "s3",
        aws_access_key_id="",
        aws_secret_access_key="",
        aws_session_token="",
        region_name="us-east-1"
    )
BUCKET = "rosu-de300spring2026-bucket"

In [4]:
def load_data():
    RATINGS = "datasets/ratings.csv"
    USERS = "datasets/users.dat"
    MOVIES = "datasets/movies.dat"

    users_df = ""
    movies_df = ""
    ratings_df = ""
    
    try:
        # Get users data
        response = s3.get_object(Bucket=BUCKET, Key=USERS)
        print("Users data exists in S3 bucket")
        print("Retrieving users data from S3 bucket")
        users_df = pd.read_csv(BytesIO(response["Body"].read()), sep="::", engine="python", encoding="latin-1", header=None, names=["user_id", "gender", "age", "occupation", "zip_code"])
    
        # Get movies data
        response = s3.get_object(Bucket=BUCKET, Key=MOVIES)
        print("Movies data exists in S3 bucket")
        print("Retrieving movies data from S3 bucket")
        movies_df = pd.read_csv(BytesIO(response["Body"].read()), sep="::", engine="python", encoding="latin-1", header=None, names=["item_id", "title", "genres"])
        
        # Get ratings data
        response = s3.get_object(Bucket=BUCKET, Key=RATINGS)
        print("Ratings data exists in S3 bucket")
        print("Retrieving ratings data from S3 bucket")
        ratings_df = pd.read_csv(response["Body"])
        
        return users_df, movies_df, ratings_df
    except:
        print("Some data not found in S3 bucket")
        dataset = Movielens(version='1m').prepare_and_load()
        ratings_df = dataset.data
        ratings_df.to_csv("ratings.csv", index=False)
        print("Uploading ratings data to S3 bucket")
        s3.upload_file("ratings.csv", BUCKET, "datasets/ratings.csv")
        
        return users_df, movies_df, ratings_df

In [5]:
def add_year_column(movies_df):
    movies_df["year"] = movies_df["title"].str.extract(r"\((\d{4})\)").astype("Int64")
    return movies_df

In [6]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_NAME = "distilbert-base-uncased"  # for illustration, 66M model

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
encoder = AutoModel.from_pretrained(MODEL_NAME).to(DEVICE)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [7]:
# BERT encode helper
@torch.no_grad()
def bert_embed(texts, max_len=32):
    batch = tokenizer(
        texts, padding=True, truncation=True, max_length=max_len, return_tensors="pt"
    )
    batch = {k: v.to(DEVICE) for k, v in batch.items()}
    out = encoder(**batch)
    cls = out.last_hidden_state[:, 0]          # first column [CLS]-like token for classification
    emb = F.normalize(cls, dim=-1)             # normalization
    return emb.cpu().numpy().astype("float32") # size (B, 768)

In [8]:
# batch size BERT encode helper
@torch.no_grad()
def bert_embed(texts, max_len=32, batch_size=4):
    all_embs = []

    for i in range(0, len(texts), batch_size):
        batch_texts = texts[i:i + batch_size]
        batch = tokenizer(
            batch_texts, padding=True, truncation=True, max_length=max_len, return_tensors="pt"
        )
        batch = {k: v.to(DEVICE) for k, v in batch.items()}
        out = encoder(**batch)
        cls = out.last_hidden_state[:, 0]
        emb = F.normalize(cls, dim=-1)
        all_embs.append(emb.cpu().numpy().astype("float32"))

        del batch, out, cls, emb

        if DEVICE == "cuda":
            torch.cuda.empty_cache()

    return np.vstack(all_embs)

In [9]:
# Create movie embeddings
def create_embeddings(df, year=2026):
    movies = df[df["year"] <= year].copy()
    movie_id_list = movies["item_id"].tolist()
    movies["text"] = movies["title"] + ". " + movies["genres"]

    # Check if index file exists in S3
    try:
        response = s3.get_object(Bucket=BUCKET, Key=f"embeddings-{year}.index")
        with open("temp.index", "wb") as f:
            f.write(response["Body"].read())
        index = faiss.read_index("temp.index")
        print("Embeddings exist in S3")

    except:
        print("Embeddings don't exist in S3")
        
        movie_vecs = bert_embed(movies["text"].tolist())
        
        index = faiss.IndexFlatIP(movie_vecs.shape[1])
        index.add(movie_vecs)

        print("Uploading embedding index file to S3")
        faiss.write_index(index, "embeddings.index")
        s3.upload_file("embeddings.index", BUCKET, f"embeddings-{year}.index")
    
    return movies, movie_id_list, index

In [10]:
# Pick a cold user and a top 5% user
def pick_users(users_df, ratings_df):
    cold_user_id = max(set(users_df["user_id"].unique())) + 1

    user_counts = ratings_df.groupby("user_id").size()
    threshold = np.percentile(user_counts, 95)
    top_users = user_counts[user_counts >= threshold].index.tolist()
    top_user_id = np.random.choice(top_users)
        
    return cold_user_id, top_user_id

In [11]:
# Build user text from last N ratings
def build_user_text(user_id, ratings, movies, N=10):
    valid_movie_ids = set(movies["item_id"]) # since movies was filtered by year earlier in pipeline
    hist = (ratings[(ratings["user_id"] == user_id) & (ratings["rating"] >= 4) & (ratings["item_id"].isin(valid_movie_ids))].sort_values("timestamp").tail(N)["item_id"].tolist())
    if not hist:
        return "no history", set()
    text = movies.set_index("item_id").loc[hist, "text"].tolist()
    return " ".join(text), set(hist)

In [12]:
# Output recommendation and store file with summary on S3
def recommend_and_store(user_id, user_type, max_year, ratings, movies, movie_id_list, k=3, N=10):
    user_text, seen = build_user_text(user_id, ratings, movies, N)
    u = bert_embed([user_text])  # (1, 768)
    scores, idx = index.search(u, k + len(seen))  # keep track of what was seen by the user
    recs = []
    
    for j in idx[0]:
        iid = movie_id_list[j]
        if iid not in seen:
            recs.append(iid)
        if len(recs) == k:
            break
            
    recommended_movies = (movies[movies["item_id"].isin(recs)][["item_id", "title", "genres"]].to_dict("records"))

    user_ratings = ratings[ratings["user_id"] == user_id]

    if len(user_ratings) > 0:
        last_interaction_time = user_ratings["timestamp"].max()
        total_ratings = len(user_ratings)
        avg_rating = user_ratings["rating"].mean()
        liked_movies = len(user_ratings[user_ratings["rating"] >= 4])
    else:
        last_interaction_time = None
        total_ratings = 0
        avg_rating = None
        liked_movies = 0

    output = pd.DataFrame([{
        "User_ID": user_id,
        "User_Type": user_type,
        "Last_Interaction_Time": last_interaction_time,
        "Total_Ratings": total_ratings,
        "Average_Rating": avg_rating,
        "Liked_Movies_Rated_4_or_Higher": liked_movies,
        "Recommendation Max Year": max_year,
        "Recommended_Movies": recommended_movies
    }])

    local_file = f"recommendations_user-{user_id}.csv"
    output.to_csv(local_file, index=False)

    s3.upload_file(local_file, BUCKET, f"recommendations/{local_file}")

    return output

In [11]:
# Execution pipeline for movies up to 1980 

In [12]:
(users_df, movies_df, ratings_df) = load_data()
# print(type(users_df))
print(users_df.columns)
# print(users_df.head(5))
# print(type(movies_df))
print(movies_df.columns)
# print(type(ratings_df))
print(ratings_df.columns)

Users data exists in S3 bucket
Retrieving users data from S3 bucket
Movies data exists in S3 bucket
Retrieving movies data from S3 bucket
Ratings data exists in S3 bucket
Retrieving ratings data from S3 bucket
Index(['user_id', 'gender', 'age', 'occupation', 'zip_code'], dtype='object')
Index(['item_id', 'title', 'genres'], dtype='object')
Index(['user_id', 'item_id', 'rating', 'timestamp'], dtype='object')


In [13]:
movies_df = add_year_column(movies_df)
print(movies_df.columns)

Index(['item_id', 'title', 'genres', 'year'], dtype='object')


In [15]:
max_year = 1980
(movies_df, movie_id_list, index) = create_embeddings(movies_df, max_year)

Embeddings don't exist in S3
Uploading embedding index file to S3


In [16]:
(cold_user_id, top_user_id) = pick_users(users_df, ratings_df)
print(f"Cold user id: {cold_user_id} \nTop user id: {top_user_id}")
# ratings_df.loc[ratings_df["user_id"] == top_user_id]

Cold user id: 6041 
Top user id: 731


In [17]:
recommend_and_store(cold_user_id, "Cold User", max_year, ratings_df, movies_df, movie_id_list, k=5, N=10)

,User_ID,User_Type,Last_Interaction_Time,Total_Ratings,Average_Rating,Liked_Movies_Rated_4_or_Higher,Recommendation Max Year,Recommended_Movies
0,6041,Cold User,None,0,None,0,1980,"[{'item_id': 2150, 'title': 'Gods Must Be Craz..."


In [19]:
recommend_and_store(top_user_id, "Top User", max_year, ratings_df, movies_df, movie_id_list, k=5, N=10)

,User_ID,User_Type,Last_Interaction_Time,Total_Ratings,Average_Rating,Liked_Movies_Rated_4_or_Higher,Recommendation Max Year,Recommended_Movies
0,731,Top User,975531200,642,3.778816,392,1980,"[{'item_id': 1030, 'title': 'Pete's Dragon (19..."


In [20]:
# Execution pipeline for all movies

In [16]:
(users_df, movies_df, ratings_df) = load_data()

Users data exists in S3 bucket
Retrieving users data from S3 bucket
Movies data exists in S3 bucket
Retrieving movies data from S3 bucket
Ratings data exists in S3 bucket
Retrieving ratings data from S3 bucket


In [17]:
movies_df = add_year_column(movies_df)

In [19]:
max_year = 2026
(movies_df, movie_id_list, index) = create_embeddings(movies_df, max_year)

Embeddings don't exist in S3
Uploading embedding index file to S3


In [20]:
(cold_user_id, top_user_id) = pick_users(users_df, ratings_df)

In [21]:
recommend_and_store(cold_user_id, "Cold User", max_year, ratings_df, movies_df, movie_id_list, k=5, N=10)

,User_ID,User_Type,Last_Interaction_Time,Total_Ratings,Average_Rating,Liked_Movies_Rated_4_or_Higher,Recommendation Max Year,Recommended_Movies
0,6041,Cold User,None,0,None,0,2026,"[{'item_id': 192, 'title': 'Show, The (1995)',..."


In [22]:
recommend_and_store(top_user_id, "Top User", max_year, ratings_df, movies_df, movie_id_list, k=5, N=10)

,User_ID,User_Type,Last_Interaction_Time,Total_Ratings,Average_Rating,Liked_Movies_Rated_4_or_Higher,Recommendation Max Year,Recommended_Movies
0,148,Top User,980194240,624,3.733974,391,2026,"[{'item_id': 1460, 'title': 'That Darn Cat! (1..."


In [ ]:
# My user profile execution pipline

In [13]:
(users_df, movies_df, ratings_df) = load_data()

Users data exists in S3 bucket
Retrieving users data from S3 bucket
Movies data exists in S3 bucket
Retrieving movies data from S3 bucket
Ratings data exists in S3 bucket
Retrieving ratings data from S3 bucket


In [14]:
movies_df = add_year_column(movies_df)

In [15]:
max_year = 2026
(movies_df, movie_id_list, index) = create_embeddings(movies_df, max_year)

Embeddings exist in S3


In [20]:
try:
    response = s3.get_object(Bucket=BUCKET, Key="profiles/my-profile.csv")
    print("My profile exists in S3 bucket")
    print("Retrieving my profile from S3 bucket...")
    my_ratings_df = pd.read_csv(response["Body"])
    
except:
    my_user_id = 7777
    my_ratings_df = pd.DataFrame([
        {"user_id": my_user_id, "item_id": 1365, "rating": 5, "timestamp": 1},
        {"user_id": my_user_id, "item_id": 2706, "rating": 4, "timestamp": 2},
        {"user_id": my_user_id, "item_id": 3667, "rating": 3, "timestamp": 3},
        {"user_id": my_user_id, "item_id": 3684, "rating": 4, "timestamp": 4},
        {"user_id": my_user_id, "item_id": 1881, "rating": 2, "timestamp": 5},
        {"user_id": my_user_id, "item_id": 2224, "rating": 5, "timestamp": 6},
        {"user_id": my_user_id, "item_id": 730, "rating": 4, "timestamp": 7},
        {"user_id": my_user_id, "item_id": 355, "rating": 3, "timestamp": 8},
        {"user_id": my_user_id, "item_id": 1241, "rating": 2, "timestamp": 9},
        {"user_id": my_user_id, "item_id": 326, "rating": 5, "timestamp": 10}
    ])
    my_ratings_df.to_csv("my-profile.csv", index=False)
    print("Uploading my profile to S3...")
    s3.upload_file("my-profile.csv", BUCKET, "profiles/my-profile.csv")

My profile exists in S3 bucket
Retrieving my profile from S3 bucket...


In [21]:
recommend_and_store(my_ratings_df["user_id"][0], "My Profile", max_year, my_ratings_df, movies_df, movie_id_list, k=5, N=10)

,User_ID,User_Type,Last_Interaction_Time,Total_Ratings,Average_Rating,Liked_Movies_Rated_4_or_Higher,Recommendation Max Year,Recommended_Movies
0,7777,My Profile,10,10,3.7,6,2026,"[{'item_id': 1010, 'title': 'Love Bug, The (19..."
